# PlurVA-LLM main Colab workflow

This is the main end-to-end notebook for the repository. Run the cells from top to bottom to clone the code, authenticate with Hugging Face, build the SFT files, train the shared LoRA adapter, generate raw test probabilities, and create the final submission ZIP.

Model checkpoints and submission files are written to Google Drive so they remain available after the Colab runtime disconnects.

## Before you start

1. In Colab, select **Runtime > Change runtime type > GPU**.
2. Request/accept access to `meta-llama/Llama-3.1-8B-Instruct` on Hugging Face.
3. Add the Hugging Face token to **Colab Secrets** as `HF_LLAMA_TOKEN`. 

1. Clone the repository

In [ ]:
!git clone https://github.com/Vihindi/pluralva_shared_task.git

 2. Configure Hugging Face authentication

In [ ]:
from google.colab import userdata
import os

hf_llama_token = userdata.get("HF_LLAMA_TOKEN")
if not hf_llama_token:
    raise ValueError("Add HF_LLAMA_TOKEN to Colab Secrets before continuing.")
os.environ["HF_TOKEN"] = hf_llama_token
print("Hugging Face authentication configured; token not displayed.")

 3. Enter the repository

In [ ]:
%cd /content/pluralva_shared_task

 4. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

5. Install dependencies

In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1" transformers peft datasets accelerate

6. Build the SFT files

This reads the three processed development datasets and writes chat-formatted full-data and fold-specific files under `sft_data/`. 


In [ ]:
!!python src/build_sft_data.py \
  --processed_dir processed \
  --out_dir sft_data \
  --si_mode binary \
  --si_aux_files processed_mmlu_aux/sri_lankan.jsonl \
  --no_value_summaries

 7. Train the shared LoRA adapter

The three full SFT files train one shared multilingual adapter for two epochs. `--load_4bit` keeps the frozen base model quantized while the LoRA parameters are trained. With the current defaults, the learning rate is `1e-4`, LoRA rank is 16, physical batch size is 2, and gradient accumulation is 10 (effective batch size 20 on one GPU).

The adapter is saved to Drive. To recover from a disconnected run when checkpoints exist, rerun this command with `--resume`.

In [ ]:
!python src/train_lora.py \
  --base_model meta-llama/Llama-3.1-8B-Instruct \
  --train_files sft_data/zh_train_full.jsonl sft_data/id_train_full.jsonl sft_data/si_train_full.jsonl \
  --output_dir /content/drive/MyDrive/pluralva_outputs/Joint_Adapter \
  --epochs 2 \
  --load_4bit

8. Generate raw predictions and the initial submission

This loads the trained adapter, averages four option permutations for Chinese and Indonesian, and uses default `0.5` Sinhala binary thresholds. `--prior_tau 0.5` applies the configured development-label prior correction to the MCQ datasets. 
Raw probabilities are checkpointed after every test item in `test_details.jsonl`, so rerunning the same command resumes rather than starting over. The directory also receives `predictions.jsonl` and `predictions.zip`.

In [ ]:
!python src/make_submission.py predict \
  --model meta-llama/Llama-3.1-8B-Instruct \
  --adapter /content/drive/MyDrive/pluralva_outputs/Joint_Adapter \
  --n_perms 4 \
  --prior_tau 0.5 \
  --th_a 0.5 \
  --th_b 0.5 \
  --no_value_summaries \
  --load_4bit \
  --out_dir /content/drive/MyDrive/pluralva_outputs/Submission_predictions

9. Apply conditional Sinhala calibration 

Compose mode reuses the saved probabilities and requires no GPU. It changes only the Sinhala conversion from `p_yes_A` and `p_yes_B` into `A`, `B`, `Both`, or `0`; Chinese and Indonesian predictions are unchanged.

These thresholds are specific to the adapter and prompt that produced the details file. Retune them whenever the model or prompt changes.

In [ ]:
!python src/make_submission.py compose \
  --details /content/drive/MyDrive/pluralva_outputs/Submission_predictions/test_details.jsonl \
  --th_b 0.92 \
  --th_a_if_b_no 0.16 \
  --th_a_if_b_yes 0.50 \
  --out_dir /content/drive/MyDrive/pluralva_outputs/final_calibrated_submission


## Output files

- Shared adapter: `/content/drive/MyDrive/pluralva_outputs/Joint_Adapter/`
- Raw probabilities: `/content/drive/MyDrive/pluralva_outputs/Submission_predictions/test_details.jsonl`
- Initial submission: `/content/drive/MyDrive/pluralva_outputs/Submission_predictions/predictions.zip`
- Calibrated submission: `/content/drive/MyDrive/pluralva_outputs/final_calibrated_submission/predictions.zip`